# Table of Contents
- [100 - Free-form descriptor filtering data](1-free-form-descriptor.ipynb#free-form-descriptor-filtering-data)

## 100 - Free-form descriptor filtering data
### Question:

You are given a dataset, a 10 by 2 array or slice. The first column contains first names, and the second column contains a free-form string with multiple words. 

For example, it includes the type of player (catcher, bowler, etc.), the team they are on, and their payment status (high pay, low pay, or free agent). 

The order of these descriptors can vary. Your task is to write a function that returns the first names matching specified criteria. 

Here's how you might call the function: 
- getNames(data, "batter") 
- getNames(data, "batter", "Dodgers")
- getNames(data, "batter", "Dodgers", "low pay") 

The number of strings can vary from 0 to 3.

Test case:

In [1]:
data = [
    ["Alice", "catchers,dodgers,highpay"],
    ["Bob", "bowler,giants,freeagent"],
    ["Charlie", "batter,angels,freeagent"],
    ["David", "batter,giants,highpay"],
    ["Eve", "batter,angels,highpay"],
    ["Frank", "bowler,dodgers,freeagent"],
    ["Grace", "catchers,dodgers,highpay"],
    ["Hannah", "bowler,angels,highpay"],
    ["Ivy", "batter,giants,lowpay"],
    ["Jack", "bowler,dodgers,highpay"]
]

### Solution 1 - intuition

In [2]:
def getNames(data: list, *criteria: str) -> str:

    result = []

    for name, desc in data:
        desc_lowercase = desc.lower()

        match = True
        # Compare all criteria
        for c in criteria:
            if c.lower() not in desc_lowercase:
                match = False
                break

        if match:
            result.append(name)

    return result

### Solution 2 - improved

Instead of searching raw strings repeatedly:

Step 1: Normalize each row into structured tokens

In [3]:
def preprocess(data):
    processed = []

    for name, desc in data:
        tokens = set(desc.lower().split(","))
        processed.append((name, tokens))

    return processed



Step 2: Query using set inclusion, not restricted by order of criteria input

In [4]:
def getNames(processed_data, *criteria):
    criteria_set = set(c.lower() for c in criteria)

    result = []
    for name, tokens in processed_data:
        if criteria_set.issubset(tokens):
            result.append(name)

    return result


`Set()`'s `issubset()` method 

In [5]:
x = set().issubset(y)

NameError: name 'y' is not defined

In [ ]:
x = {"a", "b", "c"}
y = {"f", "e", "d", "c", "b", "a"}

z = x.issubset(y)

z

True

## 82 - Count Frequencies of Unique Words

**Conditions:**
- Upper/lowercase
- 

---

**Test case:**

aabbcc \
Output: {a: 2, b: 2, c: 2}

---

**Solution:**

1. **Convert all characters to lowercase**: This ensures consistency, so that words appear the same regardless of their original case.
2. **Replace non-alphanumeric characters**: Using the `tr` command, you can substitute these characters with newline characters. This effectively converts each word into its own line.
3. **Sort and count unique occurrences**: By piping the output to the `sort` and `uniq -c` commands, the shell script will automatically count the frequency of each unique word.
4. **Sort by frequency**: You can then sort the results either in ascending or descending order based on word frequency.

In [6]:
### Python Solution 1 - Basic dict (no imports)

def count_word_frequencies_basic(text: str) -> dict:
    words = text.lower().split()
    freq = {}
    for word in words:
        # Strip punctuation manually
        word = ''.join(c for c in word if c.isalpha())
        if word:
            freq[word] = freq.get(word, 0) + 1
    return freq

count_word_frequencies_basic("The cat sat on the mat. The cat is fat.")

{'the': 3, 'cat': 2, 'sat': 1, 'on': 1, 'mat': 1, 'is': 1, 'fat': 1}

**How it works:**
- `.lower()` — normalize case
- `.split()` — split on whitespace
- `''.join(c for c in word if c.isalpha())` — strip punctuation character by character
- `dict.get(word, 0) + 1` — increment count, defaulting to 0 if key not seen yet

### Python Solution 2 - collections.Counter (idiomatic)

In [7]:
import re
from collections import Counter

def count_word_frequencies_counter(text: str) -> Counter:
    words = re.findall(r'[a-z]+', text.lower())
    return Counter(words)

freq = count_word_frequencies_counter("The cat sat on the mat. The cat is fat.")
print(freq)
print(freq.most_common(3))  # Top 3 most frequent

Counter({'the': 3, 'cat': 2, 'sat': 1, 'on': 1, 'mat': 1, 'is': 1, 'fat': 1})
[('the', 3), ('cat', 2), ('sat', 1)]


**How it works:**
- `re.findall(r'[a-z]+', text.lower())` — one step: lowercase + extract only alphabetic tokens, discarding punctuation automatically
- `Counter(words)` — builds the frequency dict in one pass
- `Counter` is a subclass of `dict`, so you can use it the same way
- `.most_common(n)` — returns top n words sorted by frequency descending; omit n for all

**Counter vs basic dict:**
- Both are O(n) time and space
- `Counter` is stdlib (no pip install), well-known, and has extras like `.most_common()` and arithmetic between counters
- Use basic dict if the interviewer says "no imports" or you want to show the underlying logic

### Bash Shell Script Solution

```bash
echo "The cat sat on the mat. The cat is fat." \
  | tr '[:upper:]' '[:lower:]' \
  | tr -cs '[:alpha:]' '\n' \
  | sort \
  | uniq -c \
  | sort -rn
```

**How each pipe step works:**

| Step | Command | What it does |
|------|---------|--------------|
| 1 | `tr '[:upper:]' '[:lower:]'` | Converts all uppercase to lowercase |
| 2 | `tr -cs '[:alpha:]' '\n'` | Replaces any non-letter character with a newline; `-c` = complement (non-alpha), `-s` = squeeze repeated delimiters |
| 3 | `sort` | Sorts words alphabetically so identical words are adjacent |
| 4 | `uniq -c` | Counts consecutive duplicate lines; `-c` prepends the count |
| 5 | `sort -rn` | Sorts numerically (`-n`) in reverse (`-r`) so highest frequency is first |

**Example output:**
```
  3 the
  2 cat
  1 fat
  1 is
  1 mat
  1 on
  1 sat
```

**Why `sort` before `uniq`?**
`uniq` only collapses *adjacent* duplicate lines — it does not group all occurrences across the whole file. You must sort first so all instances of the same word are next to each other.